# Day 65: Activation & Loss Functions

## Common Activation Functions
An activation function is applied to each neuron's output and is what gives neural networks their power: without one, stacking any number of Dense layers would still mathematically reduce to a single linear transformation, unable to learn curved decision boundaries or complex patterns. `ReLU` (Rectified Linear Unit, `max(0, x)`) is the default choice for hidden layers in most modern networks because it's cheap to compute and avoids a problem called vanishing gradients better than older alternatives — though it can occasionally cause "dead" neurons that always output 0. `Sigmoid` squashes any input into the range (0, 1), which made it historically popular for hidden layers but it's now mostly reserved for output layers of binary classifiers, since it saturates (flattens) for large positive/negative inputs and slows learning. `Tanh` is similar to sigmoid but squashes into (-1, 1) and is zero-centered, which can help gradients flow better than plain sigmoid in hidden layers. The plot below shows all three shapes side by side over an input range from -5 to 5, so you can see the flat "saturating" regions of sigmoid/tanh versus ReLU's sharp corner at zero.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-5, 5, 100)
relu = np.maximum(0, x)          # ReLU: 0 for negative inputs, identity (x) for positive inputs
sigmoid = 1 / (1 + np.exp(-x))   # Sigmoid: squashes any real number into (0, 1)
tanh = np.tanh(x)                # Tanh: squashes any real number into (-1, 1), zero-centered

plt.plot(x, relu, label='ReLU')
plt.plot(x, sigmoid, label='Sigmoid')
plt.plot(x, tanh, label='Tanh')
plt.legend()
plt.title('Activation Functions')
plt.show()

## Choosing an Output Activation
Unlike hidden layers (where ReLU is a safe default), the output layer's activation function must match the type of prediction task, because it directly determines what range and shape of values the network can produce. For binary classification, `sigmoid` outputs a single probability between 0 and 1 (the probability of the positive class). For multi-class classification, `softmax` outputs a vector of probabilities across all classes that sums to exactly 1, so you can interpret each entry as "how likely is this class." For regression, you typically use no activation at all (a "linear" output) because the target can be any real number, and squashing it into a bounded range like (0, 1) would make it impossible for the network to predict values outside that range. Getting this wrong — e.g. using sigmoid for a regression task — silently caps what the network can ever predict, so it's one of the most important architecture choices to get right.

In [ ]:
# Quick reference: which output activation + loss pairing to use for each task type
print('Binary classification -> sigmoid + binary_crossentropy')
print('Multi-class classification -> softmax + categorical_crossentropy')
print('Regression -> linear (no activation) + mse')

## Loss Functions in Keras
The loss function is the single number the optimizer is trying to minimize during training — it quantifies "how wrong" the network's current predictions are, and its gradient (with respect to each weight) is exactly what drives every weight update via backpropagation. Choosing the right loss matters as much as choosing the right output activation, and the two are tightly linked. This cell defines two separate small models to illustrate the two most common pairings: a regression model whose output layer has no activation and is trained with `mse` (Mean Squared Error — penalizes larger errors disproportionately more, since errors are squared), and a binary classification model whose output layer uses `sigmoid` and is trained with `binary_crossentropy` (which penalizes confident wrong predictions much more heavily than uncertain ones, unlike MSE).

In [ ]:
# pip install tensorflow
from tensorflow import keras
from tensorflow.keras import layers

# Regression setup: no activation on the final Dense(1) layer (output can be any real number),
# trained with Mean Squared Error
regression_model = keras.Sequential([layers.Dense(8, activation='relu', input_shape=(3,)), layers.Dense(1)])
regression_model.compile(optimizer='adam', loss='mse')

# Binary classification setup: sigmoid squashes the single output into a 0-1 probability,
# trained with binary cross-entropy (the standard loss for this pairing)
binary_model = keras.Sequential([layers.Dense(8, activation='relu', input_shape=(3,)), layers.Dense(1, activation='sigmoid')])
binary_model.compile(optimizer='adam', loss='binary_crossentropy')

## Comparing Predicted vs Actual Loss
Watching the loss value over training epochs is the most basic but most important diagnostic in deep learning: if it's steadily decreasing, the network is learning something from the data; if it's flat or erratic, something is likely wrong (learning rate too high/low, bad data, wrong loss function for the task). Here we generate a simple synthetic binary classification task where the label is just whether the first feature is positive — an easy pattern for the network to learn — and train the `binary_model` defined above for 10 epochs. Printing `history.history['loss']` per epoch lets you directly see this trajectory as numbers rather than a plot; you should expect the values to generally decrease from the first epoch to the last as the model's predictions get better.

In [ ]:
import numpy as np
# Simple synthetic binary task: label is 1 if the first feature is positive, else 0 -- an
# easy, clearly-learnable pattern so we can watch the loss actually go down
X_demo = np.random.randn(100, 3)
y_demo = (X_demo[:, 0] > 0).astype(int)
history = binary_model.fit(X_demo, y_demo, epochs=10, verbose=0)
# history.history['loss'] holds the training loss recorded at the end of each epoch
print('Loss per epoch:', [round(l, 3) for l in history.history['loss']])